# Experiment: SpectralBio Failure-Mode Validation (H100)

This notebook validates the single shortlisted failure-mode candidate from the T4 screen on a carefully bounded H100 budget.

## Why this notebook exists
- The T4 screen only ranks hypotheses; it does not create a fresh stronger-backbone validation.
- The H100 budget should only be spent if the candidate is strong enough to justify it.
- The goal is not to rerun whole genes again; the goal is to test whether the shortlisted regime keeps a stronger covariance-heavy signature under a stronger protein LM backbone.

## Success criteria
- Respect the gate from notebook 7 and stop early if the candidate is weak.
- Re-score only the validation pool, not the whole benchmark.
- Package a validation summary, examples table, figures, and zip in one pass.


In [ ]:
from pathlib import Path

USE_GOOGLE_DRIVE = False
DRIVE_MOUNT_POINT = Path("/content/drive")
DRIVE_OUTPUT_SUBDIR = Path("MyDrive/SpectralBioFailureModeValidation")
SAVE_RESULTS_ZIP = True

if USE_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount(str(DRIVE_MOUNT_POINT))
    print(f"Drive mounted at {DRIVE_MOUNT_POINT}")
else:
    print("Drive mount skipped; outputs stay in the runtime unless OUTPUT_ROOT is changed later.")

print("ACABEI PODE IR PARA O PR?XIMO")


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = os.environ.get("SPECTRALBIO_REPO_URL", "https://github.com/DaviBonetto/SpectralBio.git")
REPO_BRANCH = os.environ.get("SPECTRALBIO_REPO_BRANCH", "codex/claw4s-rebuild")
DEFAULT_REPO_ROOT = Path("/content/Stanford-Claw4s")
ENV_REPO_ROOT = os.environ.get("SPECTRALBIO_REPO_ROOT", "").strip()

def _looks_like_repo(path: Path) -> bool:
    return (path / "src" / "spectralbio").exists() and (path / "notebooks").exists()

candidate_roots = []
if ENV_REPO_ROOT:
    candidate_roots.append(Path(ENV_REPO_ROOT).expanduser())
candidate_roots.extend([Path.cwd(), Path.cwd().parent, DEFAULT_REPO_ROOT])
REPO_ROOT = next((path.resolve() for path in candidate_roots if _looks_like_repo(path)), DEFAULT_REPO_ROOT)
BOOTSTRAP_MARKER = REPO_ROOT / ".colab_bootstrap_failure_mode_validation_h100_complete"

if not _looks_like_repo(REPO_ROOT):
    REPO_ROOT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(REPO_ROOT)])

os.chdir(REPO_ROOT)
subprocess.run(["git", "fetch", "origin", REPO_BRANCH], check=False)
subprocess.run(["git", "checkout", REPO_BRANCH], check=False)

src_path = REPO_ROOT / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

if not BOOTSTRAP_MARKER.exists():
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchvision"], check=False)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", ".", "--no-deps"])
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "numpy==2.1.3",
            "pandas==2.2.3",
            "matplotlib==3.9.2",
            "scipy==1.14.1",
            "scikit-learn==1.5.2",
            "transformers==4.49.0",
            "accelerate>=1.0.0",
        ]
    )
    BOOTSTRAP_MARKER.write_text("ok\n", encoding="utf-8")
    print("Dependencies installed without restarting the runtime.")
else:
    print("Bootstrap marker found; skipping reinstall.")

print("REPO_ROOT =", REPO_ROOT)
print("Python =", sys.version.split()[0])
print("ACABEI PODE IR PARA O PR?XIMO")


## Plan

1. Load the screen summary and stop early unless the candidate is H100-worthy.
2. Re-score only the validation pool on a stronger backbone.
3. Compare candidate positives against matched pathogenic controls and benign regime variants.
4. Save figures, summary tables, and a zip bundle.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import FileLink, display

from spectralbio.constants import WINDOW_RADIUS
from spectralbio.supplementary.reject_recovery import _ensure_gene_score_rows
from spectralbio.utils.io import ensure_dir

OUTPUT_ROOT = (
    DRIVE_MOUNT_POINT / DRIVE_OUTPUT_SUBDIR
    if USE_GOOGLE_DRIVE
    else REPO_ROOT / "supplementary" / "failure_mode_validation_h100"
)
ARTIFACT_ROOT = OUTPUT_ROOT / "failure_mode_validation"
TABLES_DIR = ARTIFACT_ROOT / "tables"
FIGURES_DIR = ARTIFACT_ROOT / "figures"
SCORES_DIR = ARTIFACT_ROOT / "scores"

for directory in (ARTIFACT_ROOT, TABLES_DIR, FIGURES_DIR, SCORES_DIR):
    ensure_dir(directory)

VALIDATION_MODEL_NAME = "facebook/esm2_t33_650M_UR50D"
SCREEN_SUMMARY_PATH_OVERRIDE = os.environ.get("SPECTRALBIO_FAILURE_SCREEN_SUMMARY", "").strip()
FORCE_RUN_HEAVY = os.environ.get("SPECTRALBIO_FORCE_FAILURE_VALIDATION", "").strip().lower() in {"1", "true", "yes", "y", "on"}
OVERWRITE = False
CHECKPOINT_EVERY = 2

def resolve_existing_path(raw: str | Path) -> Path:
    raw_path = Path(raw)
    if raw_path.exists():
        return raw_path
    raw_text = str(raw).replace("\\", "/")
    for prefix in (
        "/content/Stanford-Claw4s/",
        "/teamspace/studios/this_studio/Stanford-Claw4s/",
    ):
        if raw_text.startswith(prefix):
            candidate = REPO_ROOT / raw_text[len(prefix):]
            if candidate.exists():
                return candidate
    repo_marker = "Stanford-Claw4s/"
    if repo_marker in raw_text:
        candidate = REPO_ROOT / raw_text.split(repo_marker, 1)[1]
        if candidate.exists():
            return candidate
    if not raw_path.is_absolute():
        candidate = (REPO_ROOT / raw_path).resolve()
        if candidate.exists():
            return candidate
    return raw_path

def minmax_normalize(series: pd.Series) -> pd.Series:
    values = series.astype(float)
    minimum = float(values.min())
    maximum = float(values.max())
    if maximum <= minimum:
        return pd.Series(np.zeros(len(values), dtype=float), index=series.index)
    return (values - minimum) / (maximum - minimum)

print("validation model =", VALIDATION_MODEL_NAME)
print("FORCE_RUN_HEAVY =", FORCE_RUN_HEAVY)
print("output root =", ARTIFACT_ROOT)
print("ACABEI PODE IR PARA O PR?XIMO")


In [ ]:
summary_candidates = []
if SCREEN_SUMMARY_PATH_OVERRIDE:
    summary_candidates.append(Path(SCREEN_SUMMARY_PATH_OVERRIDE))
summary_candidates.append(
    REPO_ROOT / "supplementary" / "failure_mode_screen_t4" / "failure_mode_screen" / "tables" / "candidate_failure_mode_summary.json"
)
summary_candidates.append(
    REPO_ROOT / "notebooks" / "Results 7,8,9" / "failure_mode_screen_bundle" / "tables" / "candidate_failure_mode_summary.json"
)
if USE_GOOGLE_DRIVE:
    summary_candidates.append(
        DRIVE_MOUNT_POINT / Path("MyDrive/SpectralBioFailureModeScreen/failure_mode_screen/tables/candidate_failure_mode_summary.json")
    )
summary_candidates += sorted(REPO_ROOT.glob("supplementary/**/candidate_failure_mode_summary.json"))
summary_candidates += sorted(REPO_ROOT.glob("notebooks/**/candidate_failure_mode_summary.json"))
screen_summary_path = None
for candidate in summary_candidates:
    resolved = resolve_existing_path(candidate)
    if resolved.exists():
        screen_summary_path = resolved
        break
if screen_summary_path is None or not screen_summary_path.exists():
    raise FileNotFoundError("Could not find candidate_failure_mode_summary.json. Run notebook 7 first and keep its outputs available.")

screen_summary = json.loads(screen_summary_path.read_text(encoding="utf-8"))
validation_pool_path = screen_summary_path.parent / "candidate_validation_pool.csv"
validation_pool = pd.read_csv(validation_pool_path)
selected_regime = str(screen_summary["selected_regime"])
GATE_READY_FOR_H100 = bool(screen_summary.get("ready_for_h100", False))
RUN_HEAVY = bool(GATE_READY_FOR_H100 or FORCE_RUN_HEAVY)
if RUN_HEAVY and not GATE_READY_FOR_H100 and FORCE_RUN_HEAVY:
    STOP_REASON = "Manual override enabled; running H100 despite the notebook 7 gate."
elif RUN_HEAVY:
    STOP_REASON = None
else:
    STOP_REASON = str(screen_summary.get("selection_reason", "Gate failed in notebook 7."))

panel_manifest_path = resolve_existing_path(screen_summary["panel_manifest_path"])
panel_manifest = json.loads(panel_manifest_path.read_text(encoding="utf-8"))

print("screen summary =", screen_summary_path)
print("selected_regime =", selected_regime)
print("validation_pool roles =", validation_pool["validation_role"].value_counts().to_dict())
print("GATE_READY_FOR_H100 =", GATE_READY_FOR_H100)
print("RUN_HEAVY =", RUN_HEAVY)
if STOP_REASON:
    print("STOP_REASON =", STOP_REASON)
print("ACABEI PODE IR PARA O PR?XIMO")


In [ ]:
def load_gene_sequence(gene: str) -> str:
    payload = panel_manifest["genes"].get(gene.upper())
    if payload is None:
        raise KeyError(f"Gene {gene} is not present in the panel manifest.")
    sequence_path = resolve_existing_path(payload["sequence_path"])
    return "".join(line.strip() for line in sequence_path.read_text(encoding="utf-8").splitlines() if line and not line.startswith(">"))

if not RUN_HEAVY:
    validation_summary = {
        "selected_regime": selected_regime,
        "validated": False,
        "executed_heavy_stage": False,
        "forced_run": bool(FORCE_RUN_HEAVY and not GATE_READY_FOR_H100),
        "reason": STOP_REASON,
        "validation_model_name": VALIDATION_MODEL_NAME,
        "n_validation_rows": int(validation_pool.shape[0]),
    }
else:
    rescored_frames = []
    for gene, gene_pool in validation_pool.groupby("gene", sort=True):
        sequence = load_gene_sequence(gene)
        variant_rows = gene_pool[["gene", "name", "position", "wt_aa", "mut_aa", "label"]].to_dict("records")
        score_rows = _ensure_gene_score_rows(
            gene=gene,
            sequence=sequence,
            variants=variant_rows,
            model_name=VALIDATION_MODEL_NAME,
            output_dir=SCORES_DIR / gene.lower(),
            window_radius=WINDOW_RADIUS,
            checkpoint_every=CHECKPOINT_EVERY,
            overwrite=OVERWRITE,
        )
        frame = pd.DataFrame(score_rows)
        frame["gene"] = frame["gene"].str.upper()
        frame = frame.merge(gene_pool, on=["gene", "name", "position", "wt_aa", "mut_aa", "label"], how="left")
        rescored_frames.append(frame)

    validation_df = pd.concat(rescored_frames, ignore_index=True)
    validation_df["ll_norm_650m"] = validation_df.groupby("gene")["ll_proper"].transform(minmax_normalize)
    validation_df["frob_norm_650m"] = validation_df.groupby("gene")["frob_dist"].transform(minmax_normalize)
    validation_df["pair_norm_650m"] = (0.55 * validation_df["frob_norm_650m"]) + (0.45 * validation_df["ll_norm_650m"])

    candidate_mask = validation_df["validation_role"].isin(["candidate_positive_existing_aug", "candidate_positive_reference_extension"])
    control_mask = validation_df["validation_role"].eq("matched_positive_control")
    benign_mask = validation_df["validation_role"].eq("regime_benign")
    candidate_group = validation_df[candidate_mask].copy()
    control_group = validation_df[control_mask].copy()
    benign_group = validation_df[benign_mask].copy()

    from spectralbio.pipeline.evaluate import _roc_auc_score

    def auc_between_groups(group_a: pd.DataFrame, group_b: pd.DataFrame, column: str):
        if group_a.empty or group_b.empty:
            return None
        y_true = [1] * len(group_a) + [0] * len(group_b)
        scores = group_a[column].astype(float).tolist() + group_b[column].astype(float).tolist()
        return float(_roc_auc_score(y_true, scores))

    candidate_vs_control = {
        "n_candidate": int(candidate_group.shape[0]),
        "n_control": int(control_group.shape[0]),
        "auc_frob_norm_650m": auc_between_groups(candidate_group, control_group, "frob_norm_650m"),
        "auc_ll_norm_650m": auc_between_groups(candidate_group, control_group, "ll_norm_650m"),
        "auc_pair_norm_650m": auc_between_groups(candidate_group, control_group, "pair_norm_650m"),
        "candidate_mean_frob_norm_650m": float(candidate_group["frob_norm_650m"].mean()) if not candidate_group.empty else None,
        "control_mean_frob_norm_650m": float(control_group["frob_norm_650m"].mean()) if not control_group.empty else None,
        "candidate_mean_ll_norm_650m": float(candidate_group["ll_norm_650m"].mean()) if not candidate_group.empty else None,
        "control_mean_ll_norm_650m": float(control_group["ll_norm_650m"].mean()) if not control_group.empty else None,
    }
    candidate_vs_benign = {
        "n_candidate": int(candidate_group.shape[0]),
        "n_benign": int(benign_group.shape[0]),
        "auc_frob_norm_650m": auc_between_groups(candidate_group, benign_group, "frob_norm_650m"),
        "auc_ll_norm_650m": auc_between_groups(candidate_group, benign_group, "ll_norm_650m"),
        "auc_pair_norm_650m": auc_between_groups(candidate_group, benign_group, "pair_norm_650m"),
    }
    validated = (
        candidate_vs_control["auc_frob_norm_650m"] is not None
        and candidate_vs_control["auc_ll_norm_650m"] is not None
        and candidate_vs_control["auc_frob_norm_650m"] >= 0.65
        and candidate_vs_control["auc_frob_norm_650m"] >= candidate_vs_control["auc_ll_norm_650m"] + 0.05
    )
    validation_summary = {
        "selected_regime": selected_regime,
        "validated": bool(validated),
        "executed_heavy_stage": True,
        "forced_run": bool(FORCE_RUN_HEAVY and not GATE_READY_FOR_H100),
        "reason": "Candidate re-scored on a stronger backbone." if validated else "Stronger-backbone re-score did not produce a sufficiently clean separation.",
        "validation_model_name": VALIDATION_MODEL_NAME,
        "n_validation_rows": int(validation_df.shape[0]),
        "candidate_vs_control": candidate_vs_control,
        "candidate_vs_benign": candidate_vs_benign,
        "role_counts": validation_df["validation_role"].value_counts().to_dict(),
    }
    validation_df.to_csv(TABLES_DIR / "failure_mode_validation_rows.csv", index=False)
    highlight_columns = ["gene", "name", "validation_role", "label", "ll_norm_650m", "frob_norm_650m", "pair_norm_650m", "reference_advantage", "rescue_margin"]
    validation_df[highlight_columns].sort_values(["validation_role", "gene", "pair_norm_650m"], ascending=[True, True, False]).to_csv(TABLES_DIR / "failure_mode_examples.csv", index=False)
    plt.figure(figsize=(7.5, 6.0))
    colors = validation_df["validation_role"].map({
        "candidate_positive_existing_aug": "#b42318",
        "candidate_positive_reference_extension": "#b42318",
        "matched_positive_control": "#2459a6",
        "regime_benign": "#2f7d32",
    }).fillna("#777777")
    plt.scatter(validation_df["ll_norm_650m"], validation_df["frob_norm_650m"], c=colors, alpha=0.78, s=42)
    plt.xlabel("650M normalized likelihood")
    plt.ylabel("650M normalized covariance displacement")
    plt.title("Failure-mode validation subset")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "failure_mode_validation_scatter.png", dpi=180, bbox_inches="tight")
    plt.close()

(TABLES_DIR / "failure_mode_validation_summary.json").write_text(json.dumps(validation_summary, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(validation_summary, indent=2, ensure_ascii=False))
print("ACABEI PODE IR PARA O PR?XIMO")


In [ ]:
import shutil

ZIP_BASE = ARTIFACT_ROOT / "failure_mode_validation_bundle"
zip_path = Path(shutil.make_archive(str(ZIP_BASE), "zip", ARTIFACT_ROOT))
print("Bundle written to", zip_path)
display(FileLink(str(zip_path)))

try:
    from google.colab import files

    files.download(str(zip_path))
    print("Auto-download requested through google.colab.files.download.")
except Exception as error:
    print("Auto-download was not triggered:", error)

print("ACABEI PODE IR PARA O PR?XIMO")
